# 03 · FunnyBirds + MCBM — does minimality fix grounding?

**Claim (MCBM).** The information bottleneck makes each `z_j` a *minimal sufficient
statistic* of `c_j` — and that this is what makes concepts faithful. Loss:
`L = CE(y,ŷ) + λ_c·BCE(z,c) + γ·0.2·mean((6c−3 − z)²)`; γ is the minimality knob
(**effective force = γ×0.2**). We sweep γ∈{0,…,5} (paper's range: CUB 0.05–0.3, synth 1–5).

**Hypothesis (refutation).** Minimality constrains what `z` *contains*; backwash is
about what `z` *reads*. When `c=f(class)` (notebook 01), the ±3 target is class-derived,
so tightening γ can't remove class-reading. Prediction: `retained_frac` **flat/rising**
in γ, MCBM ≈ CBM.

**γ=0** is MCBM with no IB term — *not* vanilla CBM (different head); CBM is a separate
reference line. *Reference: `fb_mcbm_renderer_swap.ipynb`, `fb_mcbm_rl_renderer_swap.ipynb`.*

In [ ]:
import os, json, re, glob
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
CURATED = Path(os.environ["CURATED_DATA"]); REPO = Path.cwd().parent
import sys; sys.path.insert(0, str(REPO/"analysis"))
try:
    from plotting import set_paper_style, PALETTE; set_paper_style()
    CBM_C, MCBM_C = PALETTE["CBM"], PALETTE["MCBM"]
except Exception:
    CBM_C, MCBM_C = "#0072B2", "#D55E00"
plt.rcParams["figure.dpi"]=120
def parse_stem(stem):
    m=re.match(r"^funnybirds-(vanilla|cbm|mcbm)(?:-g([0-9p]+))?-s(\d+)$", stem)
    if not m: return None
    gamma=float(m.group(2).replace("p",".")) if m.group(2) else np.nan
    return m.group(1), gamma, int(m.group(3))
def need(p, how):
    ok=Path(p).exists()
    if not ok: print(f"[pending] {p}\n  produce it:  {how}")
    return ok


## 1 · Overall `retained_frac` vs γ — the summary (diluted)
`backwash_vs_gamma.csv`. This averages all 5 parts, so it sits near 0.1 regardless
(4 parts ~0, only tail high) — §2 is the one to show. CBM drawn as a reference line.

In [ ]:
bw = CURATED/"backwash_vs_gamma.csv"
if need(bw, 'bash analysis/grounding_sweep.sh'):
    T = pd.read_csv(bw); display(T.round(3))
    mc = T[(T.model=="mcbm") & T.retained_frac.notna()].copy(); cb = T[T.model=="cbm"]
    g = mc.groupby("gamma").retained_frac.agg(["mean","std"]).reset_index()
    floor = (g.gamma[g.gamma>0].min() or 0.05)/3
    fig,ax=plt.subplots(figsize=(6.2,4))
    ax.errorbar(g.gamma.replace(0,floor), g["mean"], yerr=g["std"].fillna(0), marker="o", capsize=3,
                color=MCBM_C, label="MCBM (γ sweep)")
    if len(cb): ax.axhline(cb.retained_frac.mean(), ls="--", color=CBM_C, label="CBM (ref)")
    ax.set_xscale("log"); ax.set_xlabel("γ  (minimality; effective force = γ×0.2)")
    ax.set_ylabel("overall retained_frac"); ax.set_ylim(0,1.02)
    ax.set_title("Overall removed-part retention vs γ  (FunnyBirds)"); ax.legend()
    miss = T[T.retained_frac.isna()]
    if len(miss): print("MISSING/failed rows (re-run grounding):", list(zip(miss.model, miss.gamma)))

## 2 · Per-part `retained_frac` vs γ — **the figure for the meeting**
Read the per-model grounding parquets directly and break retention out **by part**.
The question: does the **tail** curve come down as γ (minimality) increases?

In [ ]:
def per_part(f):
    d = pd.read_parquet(f)
    gg = d.groupby("part").agg(pi=("p_intact","mean"), pr=("p_removed","mean"))
    return (gg.pr/gg.pi).rename("retained_frac")
recs=[]
for f in sorted(glob.glob(str(CURATED/"grounding"/"funnybirds-*-s1.parquet"))):
    pr = parse_stem(Path(f).stem)
    if pr is None or pr[0]=="vanilla": continue
    model, gamma, _ = pr
    try: s = per_part(f)
    except Exception as e: print("skip", f, e); continue
    for part, v in s.items(): recs.append(dict(model=model, gamma=gamma, part=part, retained_frac=float(v)))
P = pd.DataFrame(recs)
if len(P):
    parts=["tail","wing","beak","foot","eye"]
    cbm_pp = P[P.model=="cbm"].set_index("part").retained_frac
    mc = P[(P.model=="mcbm") & P.retained_frac.notna()]
    floor=(mc.gamma[mc.gamma>0].min() or 0.05)/3
    fig,ax=plt.subplots(figsize=(7,4.4)); cmap=plt.cm.viridis(np.linspace(0,0.85,len(parts)))
    for part,col in zip(parts,cmap):
        s=mc[mc.part==part].sort_values("gamma")
        if not len(s): continue
        ax.plot(s.gamma.replace(0,floor), s.retained_frac, "o-", color=col,
                lw=3 if part=="tail" else 1.4, label=part+("  ← backwash-prone" if part=="tail" else ""))
        if part in cbm_pp.index: ax.scatter([floor/1.6],[cbm_pp[part]], marker="*", s=90, color=col, zorder=5)
    ax.set_xscale("log"); ax.set_xlabel("γ  (minimality; effective force = γ×0.2)")
    ax.set_ylabel("retained_frac (removed part)"); ax.set_ylim(-0.02,1.02)
    ax.set_title("Per-part removed-part retention vs γ  (★ = CBM ref)\nminimality does not bring tail down")
    ax.legend(title="part", fontsize=8)
    print("tail retained_frac by γ:"); display(mc[mc.part=='tail'][['gamma','retained_frac']].sort_values('gamma').round(3))
else:
    print("[pending] no grounding parquets -> bash analysis/grounding_sweep.sh")

## 3 · Species-code vs γ — did the class channel survive compression?
`species_probe/funnybirds-mcbm-g*-s*.json`. If `species←c_preds` stays ≈1 as γ grows,
minimality compressed the representation without cutting the class channel.

In [ ]:
rows=[]
for f in sorted(glob.glob(str(CURATED/"species_probe"/"funnybirds-mcbm-g*-s1.json"))):
    m=re.search(r"-g([0-9p]+)-s", Path(f).name)
    if not m: continue
    S=json.loads(Path(f).read_text())
    rows.append(dict(gamma=float(m.group(1).replace("p",".")), z=S["species_from_z"]["acc"],
                     c=S["species_from_cpreds"]["acc"],
                     tail=S["species_from_part_cpreds"].get("tail",{}).get("acc",np.nan), chance=S["chance"]))
cbj = CURATED/"species_probe"/"funnybirds-cbm-s1.json"
if rows:
    D=pd.DataFrame(rows).sort_values("gamma"); display(D.round(3)); floor=(D.gamma[D.gamma>0].min() or 0.05)/3
    fig,ax=plt.subplots(figsize=(6.2,4))
    ax.plot(D.gamma.replace(0,floor), D.c, "o-", color=MCBM_C, label="species←c_preds")
    ax.plot(D.gamma.replace(0,floor), D.tail,"s--", color="#5B8C5A", label="species←tail concepts")
    if cbj.exists(): ax.axhline(json.loads(cbj.read_text())["species_from_cpreds"]["acc"], ls=":", color=CBM_C, label="CBM species←c_preds")
    ax.axhline(D.chance.iloc[0], ls=":", color="k", label="chance"); ax.set_xscale("log")
    ax.set_xlabel("γ"); ax.set_ylabel("species recoverable"); ax.set_ylim(0,1.02); ax.legend(fontsize=8)
    ax.set_title("Class channel survives minimality")
else:
    print("[pending] no MCBM species_probe json -> grounding_sweep.sh runs the probe too")

## 4 · CONTROL — did γ actually tighten the bottleneck?
Flat `retained_frac` only refutes minimality **if γ changed the representation**. Read
`z` from the saved predictions and measure how tightly it is pinned to the ±3 target
(`mean((6c−3 − z)²)` ↓ / `mean|z|` ↑ with γ = γ bit), plus fit quality.

In [ ]:
import torch
def zstats(cfg):
    pth = REPO/"external"/"minimal_cbm"/"results"/cfg/"1"/"predictions"/"epoch_100.pth"
    if not pth.exists(): return None
    d = torch.load(pth, map_location="cpu", weights_only=False)
    z, c = d["z"].float(), d["c"].float()
    if not (np.isfinite(z).all() and np.isfinite(c).all()): return None   # diverged -> skip
    yp = d["y_preds"]; ta = float((yp.argmax(-1)==d["y"]).float().mean()) if yp.ndim>1 else float((yp==d["y"]).float().mean())
    cp = d["c_preds"]; cp = cp[...,0] if cp.ndim==3 else cp
    return float(((6*c-3-z)**2).mean()), float(z.abs().mean()), ta, float(((cp>=0.5).float()==c).float().mean())
rows=[]
for g,tag in [(0,"g0"),(0.1,"g0p1"),(0.3,"g0p3"),(1,"g1"),(3,"g3"),(5,"g5")]:
    s=zstats(f"funnybirds-mcbm-{tag}")
    if s: rows.append((g,*s))
if rows:
    D=pd.DataFrame(rows,columns=["gamma","rep_loss","mean_abs_z","task_acc","concept_acc"]); display(D.round(3))
    floor=(D.gamma[D.gamma>0].min() or 0.05)/3
    fig,ax=plt.subplots(1,2,figsize=(11,3.8))
    ax[0].plot(D.gamma.replace(0,floor), D.rep_loss, "o-", color=MCBM_C, label="mean (±3−z)²")
    a0=ax[0].twinx(); a0.plot(D.gamma.replace(0,floor), D.mean_abs_z, "s--", color="#5B8C5A", label="mean|z|")
    ax[0].set_xscale("log"); ax[0].set_xlabel("γ"); ax[0].set_ylabel("minimality term (↓=pinned to ±3)")
    a0.set_ylabel("mean|z| (↑=saturated)"); ax[0].set_title("Did γ bite?")
    ax[1].plot(D.gamma.replace(0,floor), D.task_acc, "o-", label="task"); ax[1].plot(D.gamma.replace(0,floor), D.concept_acc, "s--", label="concept")
    ax[1].set_xscale("log"); ax[1].set_xlabel("γ"); ax[1].set_ylabel("val acc"); ax[1].set_ylim(0,1.02); ax[1].legend(); ax[1].set_title("Fit vs γ")
    plt.tight_layout()
    moved = (D.rep_loss.max()-D.rep_loss.min())>0.05*max(D.rep_loss.max(),1e-9) or (D.mean_abs_z.max()-D.mean_abs_z.min())>0.1
    print("VERDICT:", "γ moved the representation (minimality tightened) -> flat retained_frac is a real refutation"
          if moved else "γ barely moved the representation -> sweep underpowered; WIDEN γ before concluding")
else:
    print("[pending] need results/funnybirds-mcbm-g*/1/predictions/epoch_100.pth")

## Takeaway
If `retained_frac` is flat/rising in γ while the §4 control shows γ genuinely
tightened the bottleneck and §3 shows species stays recoverable, then minimality
shaped *content* but left the *source* (class channel) intact — a minimal sufficient
statistic of a class-derived label is still a class code. *Single seed (s1); error
bars need seed replication.*

## How `retained_frac` is read as a backwash measurement
No table or axis here is labelled "backwash" — the number we actually compute is the
literal **`retained_frac = P(typical concept | part removed) / P(typical concept | part
intact)`**. Reading it as backwash:

- A **grounded** concept must *see its part*. Delete the part → nothing to see → its
  probability should collapse → `retained_frac → 0`.
- A **backwashed** concept infers its part from the species / rest of the bird. Deleting
  the part changes nothing it relied on → the probability stays up → `retained_frac → 1`.

So `retained_frac` is the operational metric; "concept–class backwash" is the
*interpretation* of a high `retained_frac`. We keep the two separate so the definition
is explicit and not conflated with other uses of the word 'backwash'.